# 04 · Adversarial attacks

Optimise the object texture to flip hit->miss with PGD, CW and NES, and run counterfactual restoration.

Each attack writes snapshots and a loss history under `PGD/`, `CW/`, `NES/`.

In [ ]:
import torch
from flynet_robustness.models import get_flynet
from flynet_robustness.decoders import load_decoder

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model1, model2, idxs1, idxs2 = get_flynet('flynet_synthesized', device=device)
decoder = load_decoder(cnn=False, cnn_id='flynet_synthesized')
models = (model1, model2, idxs1, idxs2, decoder)  # FlyNet packing
cnn_id = 'flynet_synthesized'

## PGD ($L_2$)

In [ ]:
from flynet_robustness.attacks import run_pgd
out = run_pgd(models, cnn=False, cnn_id=cnn_id, norm='l2', eps=1000,
              steps=100, n_restarts=2, texttype='Black', run_id=1, device=device)
print('saved to', out)

## Carlini-Wagner and NES

In [ ]:
from flynet_robustness.attacks import run_cw, run_nes
run_cw(models, cnn=False, cnn_id=cnn_id, norm='l2', c=100, steps=150, device=device)
run_nes(models, cnn=False, cnn_id=cnn_id, sigma=6, steps=100, n_samples=15, device=device)

## Counterfactual restoration
Suppress p(hit) (illusion), then recover the nearest texture that restores classification.

In [ ]:
from flynet_robustness.attacks import counterfactual_restoration
illusion, restored, h1, h2 = counterfactual_restoration(
    models, decoder, cnn=False, cnn_id=cnn_id, steps=250, texttype='Black', device=device)
print('illusion/restored texture logits:', tuple(illusion.shape))

### CNN / RAFT targets
Pack models as `(feature_model, flynet_reference, decoder)` and set `cnn=True`, e.g. `cnn_id='0'` (vCNN), `'global_pooling'` (gCNN), `'raft_large'` (RAFT-12).